# Learning HuggingFace Text Classification

* Learning Source Site: ***Zero to Mastery***

* Tutorial: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial

* Setup: https://www.learnhuggingface.com/extras/setup

## Step on Creating Text Classification Model

1. Create and preprocess data.

2. Define the model we’d like use with `transformers.
AutoModelForSequenceClassification` (or another similar model class).

3. Define training arguments (these are hyperparameters for our model) with
`transformers.TrainingArguments`.

4. Pass TrainingArguments from 3 and target datasets to an instance of `transformers.Trainer`.

5. Train the model by calling `Trainer.train()`.

6. Save the model (to our local machine or to the Hugging Face Hub).

7. Evaluate the trained model by making and inspecting predctions on the test data.

8. Turn the model into a shareable demo.



## 2. Import necessary libraries



In [ ]:
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np

import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using touch version: {torch.__version__}")
print(f"Using datasets version: {datasets.__version__}")

## 3. Getting a dataset

Building food not food text classification model: need food not food text datasets.



In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

In [ ]:
dataset['train'][:5]

In [ ]:
#What features are they?
dataset.column_names

In [ ]:
# Access the training split
dataset['train']

### Inspect random samples

In [ ]:
import random

"""
random_indexs

random_samples
"""

random_indexs = random.sample(range(len(dataset['train'])), 5)
random_samples = dataset['train'][random_indexs]

print(random_indexs)

print("[INFO] Random samples from datasets:\n")
for text, label in zip(random_samples['text'], random_samples['label']):
  print(f"Text: {text} | Label: {label}")

In [ ]:
# Get unique label values
dataset['train'].unique('label')

In [ ]:
# Check the count of each label
from collections import Counter

Counter(dataset['train']['label'])

In [ ]:
# Turn our dataset into a DataFrame and get a random sample

food_not_food_df = pd.DataFrame(dataset['train'])
random_samples = food_not_food_df.sample(7)
random_samples

## 4. Preparing data for text classification

 1. Tokenize text: turn text into numbers (this goes from labels as well).
 2. Create a train/test split: want to train the model on the training split and want to evaluate the model on the test split.

In [ ]:
# Count each label values | Check balance or imbalance dataset
food_not_food_df["label"].value_counts()

In [ ]:
# Create mapping programmatically from dataset
idx2label = {idx: label for idx, label in enumerate(dataset['train'].unique('label')[::-1])}
label2idx = {label: idx for idx, label in idx2label.items()}

In [ ]:
print(idx2label)
print(label2idx)

In [ ]:
# Turn label to 0 and 1
def map_labels_to_numerical(example):
  example['label'] = label2idx[example['label']]

  return example

example = {'text': 'I like to eat chicken rice', 'label': 'food'}

print(map_labels_to_numerical(example))

In [ ]:
# Map the dataset labels to number (the whole datasets)
dataset = dataset['train'].map(map_labels_to_numerical)
dataset[:5]

In [ ]:
# Shuffle data and look at 5 random samples
dataset.shuffle()[:5]

### Split the dataset into training and test sets

* Train set = model will learn the patterns on this dataset
* Validation set(optional) = we can tune model's hyperparameters on this set
* Test set = model will evaluate the patterns on this dataset

In [ ]:
# Split the data into train/test set
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

In [ ]:
# Examine the training random sample
random_idx_train = random.randint(0, len(dataset['train']))
random_sample_train = dataset['train'][random_idx_train]
random_sample_train

In [ ]:
# Examine the test random sample
random_idx_test = random.randint(0, len(dataset['test']))
random_sample_test = dataset['test'][random_idx_test]
random_sample_test

### Tokenize the texts (turning texts into number)

The premise of tokenizaion is to turn words into number.

E.g. "I love you" --> [20, 58, 74]


--

* The transformers library has in-build support for Hugging Face tokenizers.

* And the class transformers.AutoTokenizer helps pair the model to the tokeinzers.

--

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path='distilbert/distilbert-base-uncased', use_fast=True)
# Use the fast implementation (on by default, note: it requires Rust installed to run in local)

tokenizer


In [ ]:
# Test out the tokenizer
# OpenAI = [40, 3021, 16585, 0]
tokenizer('I love pussy!')

* input_ids = text turned into numbers
* attention_mask = whether or not to pay attention to certain tokens (1 = yes pay attention, 0 = no do not pay attention)

In [ ]:
# Get the length of the tokenizer vocab

length_tokenizer_vocab = len(tokenizer.vocab)

# Get the maximum sequence length the tokenizer can handle
max_sequence_length = tokenizer.model_max_length

print(f"[INFO] Number of items in tokenizoer vocab : {length_tokenizer_vocab}")
print(f"[INFO] Max tokenizer input sequence length: {max_sequence_length}")

In [ ]:
# Tokenize a word
tokenizer('paine')

In [ ]:
# Check if the vocab is in the tokenizer vocab
tokenizer.vocab['pizza']

In [ ]:
# See how the word get tokenized
tokenizer.convert_ids_to_tokens(tokenizer("paine").input_ids)

In [ ]:
# Try to tokenize an emoji
tokenizer.convert_ids_to_tokens(tokenizer("❤️").input_ids)

In [ ]:
# Get the first three items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:3]

In [ ]:
# Get five random items in tokenizer vocab

random.sample(sorted(tokenizer.vocab.items()), k=5)

### Making a preprocessing function to tokenize text

Want to make it easy to go from sample -> tokenized_sample.

In [ ]:
def tokenize_text(examples):
  """
  Tokenize given text and return tokenized text.
  """

  return tokenizer(examples["text"],
                   padding=True, # pad short sequences to longest sequence in the batch
                   truncation=True) # truncate long sequences to the maximum length the model can handle

In [ ]:
example_text2 = {'text': "I love playing video game.", "label": 1}

# Test function
tokenize_text(example_text2)

In [ ]:
# Map tokenize text function into the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True, # set batched=True to operate across batches of examples rather than only single examples
                                batch_size=1000) # defaults to 1000, can be increased if you have a large dataset

tokenized_dataset

* Note: In machine learning, it is often faster to do things in batches rather than one at the time during leveraging computer hardware parallelization.

See more in the map documentation: https://huggingface.co/docs/datasets/en/process#batch-processing


In [ ]:
# Get two samples from the tokenized dataset
train_tokenized_sample = tokenized_dataset['train'][0]
test_tokenized_sample = tokenized_dataset['test'][0]

for key in train_tokenized_sample.keys():
  print(f" [INFO] Key: {key}")
  print(f" Train sample: {train_tokenized_sample[key]}")
  print(f" Test sample: {test_tokenized_sample[key]}")
  print()


### Tokenization Takeaways

1. Tokenizer = turn the data into numbers (e.g. text -> map to number)
2. Many Models are out there and have different tokenizers, Hugging Face's Auto (e.g. Autotokenizer, AutoProcessor, AutoModel, etc help to match tokenizers to models)
3. Tokenization can help in parallel using map and batched functions

## 5. Setting up an Evaluation Metrics

In [ ]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load('accuracy')

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
  Compute the accuracy of the model by comparing predictions and true labels
  """

  predictions, labels = predictions_and_labels

  # Get the highest prediction probability of each prediction if predictions are probabilities
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
# Create example list of predictions and labels
example_predictions_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_one_wrong = np.array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0])
example_labels_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

print(f"Accuracy when all predictions are correct: {compute_accuracy((example_predictions_all_correct, example_labels_all_correct))}")
print(f"Accuracy when one prediction is wrong: {compute_accuracy((example_predictions_one_wrong, example_labels_all_correct))}")

In [ ]:
# Get id and label mappings
print(f"label2id: {label2idx}")
print(f"id2label: {idx2label}")

In [ ]:
from transformers import AutoModelForSequenceClassification

# Set up model for fine-tuning with classification head (top layers of network)
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path = "distilbert/distilbert-base-uncased",
    num_labels = 2, # (foot/ not_food) can customize this to the number of classes in our own dataset
    id2label = idx2label, # mappings from class ID to class labels (for classification tasks)
    label2id = label2idx

)

In [ ]:
model

The model is comprised of the following parts:

1. `embeddings` - embeddings are a form of learned representation of tokens. So if tokens are a direct mapping from token to number, embedding are a learning vector representation.
2. `transformer` - the model architecture backbone, this has discoverd patterns/relationships in the embeddings.
3. `classifier` - we need to customize this layer to suit our problem.

Note: If you get input errors from passing a sample to a model, make sure the sample you pass to your model is formatting in the same way you model was trained on. For example, if your model used a specific tokenizer, make sure to tokenize your text before passing it to the model.

`model([102, 80, 300, 98])` wil not work✅.

### Count the parameters in the model

Weights/parameters = small numeric opportunies for a model to learn patterns in data.

In [ ]:
def count_params(model):
  """
  Count the parameters of a Pytouch model.
  """
  trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total_parameters = sum(p.numel() for p in model.parameters())

  return {"trainable parameters:": trainable_parameters, "total parameters:": total_parameters}

# Counts the parameters of the model
count_params(model)

The model that I am using has around **67M** parameters and **all** of them are trainable.

Note:
* Generally, the more parameters a model has, the more capacity it has to learn.
* For comparison models such as Llama 3 8B has 8 billion parameters.
* If you want the best possible performance, generally more parameters is better.
  * However, with more parameters requires more compute + time.
  * You'll be suprised how well a samller model can perform with specific data.



### Create a directory for saving models

In [ ]:
# Create model output directory
from pathlib import Path

# Create model directory
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = 'learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir


## Setting up training arguments with TrainingArguments

In [ ]:
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints to: {model_save_dir}")

# Create training arguments
training_args = TrainingArguments(
    output_dir = model_save_dir,
    learning_rate = 0.0001,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none",
    # push_to_hub=True,
    # hub_token="own_token_here",
    hub_private_repo=False
)

In [ ]:
# training_args

## Setting up an instance of Trainer

In [ ]:
tokenizer

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

#Set up trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test'],
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics = compute_accuracy,
)

### Training the model

In [ ]:
# Train a text classification model
result = trainer.train()

In [ ]:
# Inspect training metrics
for key, value in result.metrics.items():
  print(f"{key}: {value}")

### Save the model for later use

**Note**: if you are saving a model to Google Colab, note that it will disappear from Colab instance when it disconnects.

In [ ]:
# Save model
print(f"[INFO] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

### Inspect the model training metrics

In [ ]:
# Get training history
trainer_history_all = trainer.state.log_history
trainer_history_metrics = trainer_history_all[:-1] # get everything except the training time metrics (we've seen these already)
trainer_history_training_time = trainer_history_all[:1] # this is the same value as results.metrics from above

# View the first 4 metrics from the training history
trainer_history_metrics[:4]

In [ ]:
import pprint

# Extract training and evaluation metrics
trainer_history_eval_set = []
trainer_history_training_set = []

# Loop through metrics and filter for training and eval metrics
for item in trainer_history_metrics:
  item_keys = list(item.keys())

  # Check to see if "eval" is in the keys of the item
  if any("eval" in item for item in item_keys):
    trainer_history_eval_set.append(item)
  else:
    trainer_history_training_set.append(item)

# Show the first two items in each metric set
print("[INFO] First two items in training set:")
pprint.pprint(trainer_history_training_set[:2])

print("\n [INFO] First two items in evaluation set:")
pprint.pprint(trainer_history_eval_set[:2])

In [ ]:
# Create pandas DataFrames for the training and evaluation metrics
trainer_history_training_df = pd.DataFrame(trainer_history_training_set)
trainer_history_eval_df = pd.DataFrame(trainer_history_eval_set)

In [ ]:
trainer_history_training_df.head()

In [ ]:
trainer_history_eval_df.head()

In [ ]:
# Plot training and evaluation loss
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(trainer_history_training_df['epoch'], trainer_history_training_df['loss'], label='Training Loss')
plt.plot(trainer_history_eval_df['epoch'], trainer_history_eval_df['eval_loss'], label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Text Classification with DistilBert training and evaltion loss over time')

plt.show()

In [ ]:
# # Save Model to Hugging Face Hub
# # It will be public, since we set hub_private_repo=False in our TrainingArguments

# model_upload_url = trainer.push_to_hub(
#     commit_message = 'Uploading food not food text classification model',
#     # token="YOUR_HF_TOKEN_HERE" # This will default to the token you have saved in your Hugging Face config
# )

# print(f'[INFO] Model successfuly uploaded to Hugging Face Hub with at URL: {model_upload_url}')

## Making and evaluating predictions on the test dataset

In [ ]:
# Performs the predictions on the test set
predictions_all = trainer.predict(tokenized_dataset['test'])
predictions_values = predictions_all.predictions
predictions_metrics = predictions_all.metrics

print('[INFO] Predictions metrics on the test set.')
predictions_metrics

### Get predicted probabilities and evaluate by hand

In [ ]:
import torch
from sklearn.metrics import accuracy_score

# Get predictons probability
pred_probs = torch.softmax(torch.tensor(predictions_values), dim=1)

# Get the predicted labels
pred_labels = torch.argmax(pred_probs, dim=1)

# Get the true labels from dataset
true_labels = tokenized_dataset['test']['label']

# Compare the true labels and prediction labels using accuracy_score
test_accuracy = accuracy_score(y_true=true_labels,
                               y_pred=pred_labels)

print(f"[INFO] Test Accuracy: {test_accuracy*100}%")

**Note**: If you want a good evaluation method, make predictions on your entire test dataset, then index on the predictions which are wrong but have high prediction probability. For example, get the top 100-1000 and go through all of the examples where the model's prediction had high probability but was incorrect $\rightarrow$ this often leads to great insights into your data.

In [ ]:
idx2label

In [ ]:
label2idx

In [ ]:
# Make a DataFrame of test predictions
test_predictions_df = pd.DataFrame({
    'text': dataset['test']['text'],
    'true_label': true_labels,
    'pred_label': pred_labels,
    'pred_prob': torch.max(pred_probs, dim=1).values,
})

test_predictions_df.head(5)

In [ ]:
# Show 10 examples with low predicton probability
test_predictions_df.sort_values('pred_prob', ascending=True).head(10)

## Making and inspecting predictions on custom text data

In [ ]:
# Set up the local model
local_model_path = 'models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# Set up the Hugging Face model path
huggingface_model_path = 'SaiePaine/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

### Discussing ways to make predictions (inference)

**Note**: Whenever you hear the word "inference" it means to use a model to make predictions on data.

Two main ways to perform inference:
1. Pipeline mode - Using transformers. pipeline to load our model and perform text classification. See the docs:
https://huggingface.co/docs/transformers/en/main_classes/pipelines
2. PyTorch mode - Using a combination of transformers.AutoTokenizer and transformers.AutoModelForSequenceClassification
and passing each our target model name.

Each mode supports:
1. Predictions one at a time (fast but can be slower with many many samples).
    * Helpful for say a comment system and comments happen sporadically, to predict whether the comment was "spam" or "not spam".
2. Batches of prodictions at a time (faster but up to a point, e.g. say you predict on 32 samples at a time, this may be way faster than one at a time but if you go to 128 at a time, you may not see many more speedups).
    * Helpful for when you have a large static database or many samples coming in at once.

In [ ]:
# Setup our device for making predictions
# Note: generally the fast the hardware accelerator, the fast the predictions
# if you have a dedicated GPU, you should use over CPU.

def set_device():

  if torch.cuda.is_available():
    device = torch.device('cuda')
  elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device('mps')
  else:
    device = torch.device('cpu')
  return device

DEVICE = set_device()
print(f"[INFO] Using device: {DEVICE}")

### Making predictions with pipeline

In [ ]:
import torch
from transformers import pipeline

# Set the batch size for predictions
BATCH_SIZE = 32

# Create an instance of transformers.pipeline
food_not_food_classifier = pipeline(
    task='text-classification',
    model=local_model_path,
    device=DEVICE,
    top_k=1,
    batch_size=BATCH_SIZE
)

food_not_food_classifier

In [ ]:
# Test the training model on example text
sample_text_food = "I eat bacon and egg."
food_not_food_classifier(sample_text_food)

In [ ]:
# Test the model on some more example text
sample_text_not_food = "A F1 car driving so fast in Monaco Grand Prix."
food_not_food_classifier(sample_text_not_food)

In [ ]:
# Pass in random text to the model
food_not_food_classifier("cvnhertiejhwgdjshdfgh394587")

In [ ]:
# Pipeline with remote models

food_not_food_classifier_remote = pipeline(
    task='text-classification',
    model=huggingface_model_path,
    batch_size=BATCH_SIZE,
    device=DEVICE,

)

food_not_food_classifier_remote


In [ ]:
food_not_food_classifier_remote('This is about bananas, pancake and ice cream.')

In [ ]:
# Testing with not food caption sentences
not_food_sentences = [
    "I whipped up a fresh batch of code, but it seems to have a syntax error.",
    "We need to marinate these ideas overnight before presenting them to the client.",
    "The new software is definitely a spicy upgrade, taking some time to get used to.",
    "Her social media post was the perfect recipe for a viral sensation.",
    "He served up a rebuttal full of facts, leaving his opponent speechless.",
    "The team needs to simmer down a bit before tackling the next challenge.",
    "The presentation was a delicious blend of humor and information, keeping the audience engaged.",
    "A beautiful array of fake wax foods (shokuhin sampuru) in the front of a Japanese restaurant.",
    "Daniel Bourke is really cool :D",
    "A sleek stainless steel refrigerator standing empty in a modern kitchen showroom."
]

food_not_food_classifier_remote(not_food_sentences)

In [ ]:
# Testing with food caption sentences
food_sentences = [
    "My favourite food is biltong!",
    "I eat bacon and egg for breakfast almost every single morning.",
    "Sizzling hot pepperoni pizza with a thick layer of gooey, melted mozzarella cheese.",
    "A colorful bowl of mixed carrots, including vibrant orange and deep purple varieties.",
    "Creamy cauliflower curry served alongside warm, buttery garlic naan bread.",
    "A fresh fruit platter stacked with sliced mango, kiwi, strawberries, and pineapples.",
    "Two handfuls of ripe yellow bananas resting in a ceramic bowl on the dining table.",
    "Aromatic goat curry with tender meat pieces served over a bed of steamed basmati rice.",
    "Tangy fish curry bowl made with a zesty tamarind sauce and fresh curry leaves.",
    "A rich and spicy lamb rogan josh garnished with a dollop of creamy Greek yogurt."
]

food_not_food_classifier_remote(food_sentences)

In [ ]:
# Testing with confusing sentences
confusing_sentences = [
    # Uses explicit food names, but refers to tech/slang
    "He decided to flash the raspberry pi with a fresh Linux operating system.",
    "The security team discovered a serious data leak buried deep inside the apple ecosystem.",
    "Please don't use that sketchy website, it is completely filled with cookies and tracking scripts.",
    "She spent the afternoon squeezing juice out of her old iPhone battery to see if it would turn on.",

    # Uses heavy cooking metaphors for non-food activities
    "The corrupt politician was caught trying to cook the books before the tax audit.",
    "The defense attorney completely roasted the witness during the cross-examination.",
    "Let's just beef up our server infrastructure before the massive Black Friday traffic hits.",
    "He is a total couch potato who just stares at a monitor playing video games all day.",

    # Describes food items, but explicitly states they are inedible objects
    "A plastic replica of a pepperoni pizza sitting in a toy store window display.",
    "He bought a beautifully scented candle that smells exactly like fresh blueberry pancakes.",
    "She wore a bright yellow dress printed with tiny illustrations of avocados and lemons.",
    "The artist painted a stunning still life portrait featuring a bowl of grapes and a half-eaten loaf of bread."
]

food_not_food_classifier_remote(confusing_sentences)

### Time the model across larger sample sizes

In [ ]:
import time

# Create 1000 sentences
sentences_1000 = food_sentences * 100

# Time how long it takes to make predictions on all sentences (one at a time)
print(f"[INFO] Number of sentences: {len(sentences_1000)}")

start_time_one_at_a_time = time.time()

for sentence in sentences_1000:
  # Make a prediction on each sentence one at a time
  food_not_food_classifier_remote(sentence)

end_time_one_at_a_time = time.time()

total_time_one_at_a_time = end_time_one_at_a_time - start_time_one_at_a_time

print(f"[INFO] Time taken for one at a time prediction: {total_time_one_at_a_time} seconds")
print(f"[INFO] Average inference time per sentence: {total_time_one_at_a_time / len(sentences_1000)} seconds")

In [ ]:
for i in [100, 1000, 10_000]:
  sentences_big = food_sentences * i
  print(f"[INFO] Number of sentences: {len(sentences_big)}")

  start_time = time.time()
  # Predict on all sentences in batches
  food_not_food_classifier_remote(sentences_big)

  end_time = time.time()

  print(f"[INFO] Time taken for a batch prediction of {len(sentences_big)} sentences: {end_time - start_time} seconds")
  print(f"[INFO] Average inference time per sentence: {(end_time - start_time) / len(sentences_big)} seconds")
  print()



### Making predictions with Pytoch

In [ ]:
from transformers import AutoTokenizer

# Setup Hugging Face model path
model_path = 'SaiePaine/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# Create an example to predict on
sample_text_food = 'A delicious photo of a plate of scrambled eggs, bacon and toast'

# Prepare the tokenizer and tokenize the inputs
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)
inputs = tokenizer(sample_text_food,
                   return_tensors='pt') # return the output as PyTorch tensors

inputs

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load the Text Classificaton Model
model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path)

model

In [ ]:
import torch

# It is a more modern, faster, and more secure drop-in replacement for the older with torch.no_grad(): context manager
# with torch.no_grad():
with torch.inference_mode():
  outputs = model(**inputs)
  outputs_verbo = model(input_ids=inputs['input_ids'],
                        attention_mask=inputs['attention_mask'])
outputs

In [ ]:
outputs_verbo

In [ ]:
# Convert logit to prediction probability + label

predicted_class_id = outputs.logits.argmax().item()
prediction_probability = torch.softmax(outputs.logits, dim=1).max().item()

print(f'Text: {sample_text_food}')
print(f'Predicted Label: {model.config.id2label[predicted_class_id]}')
print(f'Prediction probability: {prediction_probability}')

In [ ]:
food_not_food_classifier_remote(sample_text_food)